# v24.1 Final Worst-case A — no model

No GPU, no generation. Uses saved option-wise debug if available; otherwise MC falls back to majority. Useful for fast offline verification.

In [ ]:

# ==================================================================
# Common utilities: paths, parsing, metrics
# ==================================================================
import os, re, json, math, glob, random, gc, time
from pathlib import Path
from collections import Counter, defaultdict

SEED = 42
random.seed(SEED)

LABELS = ["A", "B", "C", "D", "Yes", "No", "Unknown"]
MC_LABELS = {"A", "B", "C", "D"}
YNU_LABELS = {"Yes", "No", "Unknown"}

DATA_ROOT = Path('/kaggle/input/datasets/yixuanisthebest/v2333333')
CAND_PATHS = [
    DATA_ROOT / 'v23_val_candidates.json',
    Path('/kaggle/working/v23_candidate_rerank/v23_val_candidates.json'),
]
DEBUG_PATTERNS = [
    '/kaggle/input/**/v24_1_optionwise_mc_debug*.json',
    '/kaggle/working/**/v24_1_optionwise_mc_debug*.json',
]
OUT_DIR = Path('/kaggle/working/v24_1_final_worstcase_A_no_model')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Set this True for validation. For unlabeled test/final, set False and use structural MC detector.
STRICT_VAL_MC_BY_GOLD = True


def norm_answer(x):
    if x is None:
        return 'Unknown'
    s = str(x).strip()
    low = s.lower()
    if low in {'a','b','c','d'}:
        return low.upper()
    if low in {'yes','true','y'}:
        return 'Yes'
    if low in {'no','false','n'}:
        return 'No'
    if low in {'unknown','unk','cannot determine','uncertain'}:
        return 'Unknown'
    # extract final answer in messy strings
    m = re.search(r'\b(A|B|C|D|Yes|No|Unknown)\b', s, re.I)
    if m:
        return norm_answer(m.group(1))
    return 'Unknown'


def get_gold(r):
    return norm_answer(r.get('gold') or r.get('answer') or r.get('label') or r.get('target'))


def get_candidates(r):
    return r.get('candidates') or r.get('outputs') or []


def cand_answer(c):
    return norm_answer(c.get('answer') or c.get('pred') or c.get('final_answer'))


def cand_supp(c):
    sp = c.get('supporting_premises')
    if sp is None:
        sp = c.get('support') or c.get('premises') or []
    if isinstance(sp, str):
        nums = re.findall(r'\d+', sp)
        return [int(x) for x in nums]
    if isinstance(sp, list):
        out=[]
        for x in sp:
            try: out.append(int(x))
            except Exception: pass
        return out
    return []


def is_citation_valid(c):
    sp = cand_supp(c)
    return len(sp) > 0 and all(isinstance(x, int) and x > 0 for x in sp)


def is_mc_record(r):
    gold = get_gold(r)
    if gold in MC_LABELS:
        return True
    if STRICT_VAL_MC_BY_GOLD:
        return False
    # fallback for unlabeled test: look for explicit options in question/options fields
    opts = r.get('options') or r.get('choices')
    if isinstance(opts, dict):
        return all(k in opts for k in ['A','B','C','D'])
    q = str(r.get('question',''))
    return bool(re.search(r'\bA\s*[\).:]', q) and re.search(r'\bB\s*[\).:]', q))


def majority_answer(r, allowed=None):
    answers = [cand_answer(c) for c in get_candidates(r)]
    if allowed:
        filt = [a for a in answers if a in allowed]
        if filt:
            answers = filt
    if not answers:
        return 'Unknown'
    cnt = Counter(answers)
    # deterministic tie-break: candidate order, then label order
    top = max(cnt.values())
    tops = {a for a,v in cnt.items() if v == top}
    for a in answers:
        if a in tops:
            return a
    return cnt.most_common(1)[0][0]


def first_answer(r):
    cs = get_candidates(r)
    return cand_answer(cs[0]) if cs else 'Unknown'


def oracle_answer(r):
    gold = get_gold(r)
    for c in get_candidates(r):
        if cand_answer(c) == gold:
            return gold
    return majority_answer(r)


def score_yun_candidate(c, counts, n, params=None):
    # v24.1-final default: do NOT overfit Yes rescue; keep Unknown/No stable.
    params = params or {}
    ans = cand_answer(c)
    sp = cand_supp(c)
    vote = counts.get(ans, 0) / max(n, 1)
    score = 1.0 * vote
    if is_citation_valid(c):
        score += 0.25
    if 1 <= len(sp) <= 5:
        score += 0.15
    if len(sp) > 8:
        score -= 0.20
    # light class prior from v24.1: protect Unknown, avoid Yes over-rescue.
    if ans == 'Unknown':
        score += 0.08
    if ans == 'No':
        score -= 0.03
    if ans == 'Yes':
        score += 0.02
    raw = str(c.get('raw') or c.get('text') or c.get('output') or '')
    if re.search(r'contradict|not supported|not entailed', raw, re.I) and ans == 'Yes':
        score -= 0.30
    return score


def yni_rerank_answer(r):
    cs = get_candidates(r)
    if not cs:
        return 'Unknown'
    ycs = [c for c in cs if cand_answer(c) in YNU_LABELS]
    if not ycs:
        return majority_answer(r, YNU_LABELS)
    answers = [cand_answer(c) for c in ycs]
    counts = Counter(answers)
    n = len(ycs)
    best = max(ycs, key=lambda c: score_yun_candidate(c, counts, n))
    return cand_answer(best)


def parse_support_yes(text):
    if not text:
        return False
    s = str(text)
    # Accept Support: Yes, Support\nYes, Supported: Yes, Answer: Yes
    m = re.search(r'\bSupport(?:ed)?\s*[:\-]?\s*\n?\s*(Yes|No)\b', s, re.I)
    if m:
        return m.group(1).lower() == 'yes'
    m = re.search(r'\bAnswer\s*[:\-]?\s*\n?\s*(Yes|No)\b', s, re.I)
    if m:
        return m.group(1).lower() == 'yes'
    return False


def load_candidates():
    for p in CAND_PATHS:
        if p.exists():
            with open(p, 'r', encoding='utf-8') as f:
                data = json.load(f)
            print('Loaded candidates:', p, 'N=', len(data))
            return data
    raise FileNotFoundError('Could not find v23_val_candidates.json. Add dataset yixuanisthebest/v2333333 or copy file to working.')


def find_debug_path():
    hits = []
    for pat in DEBUG_PATTERNS:
        hits += glob.glob(pat, recursive=True)
    hits = sorted(set(hits))
    print('Debug candidates:', hits[:5], 'count=', len(hits))
    return Path(hits[0]) if hits else None


def load_debug_optional():
    p = find_debug_path()
    if not p:
        return None
    with open(p, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print('Loaded optionwise debug:', p, 'records=', len(data))
    return data


def optionwise_answer_from_debug(r, debug):
    rid = str(r.get('idx', r.get('id', r.get('record_idx', r.get('sample_idx', ''))))
    # Many saved debug files use VAL index string as key. Try several keys.
    keys = [rid, str(r.get('val_idx','')), str(r.get('candidate_idx',''))]
    # If result order id exists in r, use it; else fallback done by caller with enumerate.
    for k in keys:
        if k and k in debug:
            item = debug[k]
            sup = item.get('support', {})
            trues = [a for a in ['A','B','C','D'] if bool(sup.get(a))]
            if len(trues) == 1:
                return trues[0]
            if len(trues) > 1:
                # tie: candidate majority among supported options first, then A/B/C/D prior
                maj = majority_answer(r, set(trues))
                if maj in trues: return maj
                return trues[0]
            return majority_answer(r, MC_LABELS)
    return None


def optionwise_answer_by_index(i, r, debug):
    if debug is None:
        return majority_answer(r, MC_LABELS)
    for k in [str(i), str(r.get('idx','')), str(r.get('id','')), str(r.get('record_idx',''))]:
        if k in debug:
            item = debug[k]
            sup = item.get('support', {})
            trues = [a for a in ['A','B','C','D'] if bool(sup.get(a))]
            if len(trues) == 1:
                return trues[0]
            if len(trues) > 1:
                maj = majority_answer(r, set(trues))
                if maj in trues: return maj
                # prefer supported option that appears in any candidate; else first support
                cand_ans = [cand_answer(c) for c in get_candidates(r)]
                for a in cand_ans:
                    if a in trues: return a
                return trues[0]
            return majority_answer(r, MC_LABELS)
    return majority_answer(r, MC_LABELS)


def v24_1_final_answer(i, r, debug=None):
    if is_mc_record(r):
        return optionwise_answer_by_index(i, r, debug)
    return yni_rerank_answer(r)


def compute_metrics(golds, preds, name='metric'):
    n = len(golds)
    out = {'name': name, 'n': n, 'acc': sum(g==p for g,p in zip(golds,preds))/max(n,1)}
    per = {}
    f1s=[]; weights=[]
    for lab in LABELS:
        tp = sum((g==lab and p==lab) for g,p in zip(golds,preds))
        fp = sum((g!=lab and p==lab) for g,p in zip(golds,preds))
        fn = sum((g==lab and p!=lab) for g,p in zip(golds,preds))
        support = sum(g==lab for g in golds)
        pred_count = sum(p==lab for p in preds)
        prec = tp/(tp+fp) if tp+fp else 0.0
        rec = tp/(tp+fn) if tp+fn else 0.0
        f1 = 2*prec*rec/(prec+rec) if prec+rec else 0.0
        per[lab] = dict(precision=prec, recall=rec, f1=f1, support=support, pred_count=pred_count, tp=tp, fp=fp, fn=fn)
        f1s.append(f1); weights.append(support)
    out['macro_f1'] = sum(f1s)/len(f1s)
    out['weighted_f1'] = sum(f*w for f,w in zip(f1s,weights))/max(sum(weights),1)
    out['mc_macro_f1'] = sum(per[x]['f1'] for x in ['A','B','C','D'])/4
    out['ynu_macro_f1'] = sum(per[x]['f1'] for x in ['Yes','No','Unknown'])/3
    out['per_label'] = per
    return out


def print_metric(m):
    print('\n' + '='*100)
    print(m['name'])
    print('='*100)
    print(f"N={m['n']} acc={m['acc']:.3f} macro_f1={m['macro_f1']:.3f} weighted_f1={m['weighted_f1']:.3f} MC={m['mc_macro_f1']:.3f} YNU={m['ynu_macro_f1']:.3f}")
    print('-'*100)
    print(f"{'Label':<10} {'P':>7} {'R':>7} {'F1':>7} {'Gold':>6} {'Pred':>6}")
    for lab in LABELS:
        x=m['per_label'][lab]
        print(f"{lab:<10} {x['precision']:7.3f} {x['recall']:7.3f} {x['f1']:7.3f} {x['support']:6d} {x['pred_count']:6d}")


def to_jsonable(x):
    import numpy as np
    if isinstance(x, (np.integer,)): return int(x)
    if isinstance(x, (np.floating,)): return float(x)
    if isinstance(x, (np.bool_,)): return bool(x)
    if isinstance(x, dict): return {str(k): to_jsonable(v) for k,v in x.items()}
    if isinstance(x, (list, tuple)): return [to_jsonable(v) for v in x]
    return x


In [ ]:

# ==================================================================
# Evaluate v24.1-final on VAL candidates
# ==================================================================
results = load_candidates()
debug = load_debug_optional()

# Check MC count under gold-strict mode
golds = [get_gold(r) for r in results]
mc_count = sum(g in MC_LABELS for g in golds)
print('Gold distribution:', Counter(golds))
print('Gold MC count:', mc_count)

pred_first = [first_answer(r) for r in results]
pred_majority = [majority_answer(r) for r in results]
pred_oracle = [oracle_answer(r) for r in results]
pred_v241 = [v24_1_final_answer(i, r, debug) for i, r in enumerate(results)]

metrics = {
    'first': compute_metrics(golds, pred_first, 'first'),
    'majority': compute_metrics(golds, pred_majority, 'majority'),
    'oracle_candidates': compute_metrics(golds, pred_oracle, 'oracle_candidates'),
    'v24_1_final': compute_metrics(golds, pred_v241, 'v24_1_final'),
}
for m in metrics.values():
    print_metric(m)

summary = {
    'method': 'v24.1-final worstcase-A no-model',
    'notes': 'MC uses saved/generated option-wise support if available; YNU uses v24.1 stable rerank. No training.',
    'metrics': metrics,
    'gold_dist': dict(Counter(golds)),
    'pred_dist_v24_1_final': dict(Counter(pred_v241)),
    'option_debug_loaded': debug is not None,
    'option_debug_records': len(debug) if isinstance(debug, dict) else 0,
}
with open(OUT_DIR/'v24_1_final_summary.json', 'w', encoding='utf-8') as f:
    json.dump(to_jsonable(summary), f, ensure_ascii=False, indent=2)
rows=[]
for i,r in enumerate(results):
    rows.append({
        'idx': i,
        'gold': get_gold(r),
        'pred': pred_v241[i],
        'method': 'mc_optionwise' if is_mc_record(r) else 'ynu_rerank',
        'question': str(r.get('question',''))[:500],
    })
with open(OUT_DIR/'v24_1_final_val_predictions.json', 'w', encoding='utf-8') as f:
    json.dump(to_jsonable(rows), f, ensure_ascii=False, indent=2)
print('Saved:', OUT_DIR/'v24_1_final_summary.json')
print('Saved:', OUT_DIR/'v24_1_final_val_predictions.json')
